# Part 9 · Notebook 07 — Breaks, stability and multiple testing

**Sessions:** S7 (Advanced analysis: breaks, stability & multiple testing) · [Lesson plan](../../docs/lessons/PART_09_STATISTICAL_TRADING.md) · graded labs in [`labs/part09/`](../../labs/part09/)

**You will:**
1. Test for a break at a known date (Chow) and an unknown one (CUSUM), and see when they mislead.
2. Monitor a pair with rolling statistics and write an automatic retirement rule.
3. Measure what retirement saves on a pair that dies.
4. Test "the best of 50 strategies" honestly with White's Reality Check.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.
All data is synthetic with known parameters (true half-lives, true pairs, true hedge ratios, planted premia), so every estimator can be checked against the truth.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p9lib.py is in notebooks/part09/
    sys.path.insert(0, str(d))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p9lib as p

p.use_course_style()
import statsmodels.api as sm
from scipy.stats import f as f_dist

## 1. The Chow test

Is `y = α + βx` the same before and after a **known** date? Fit it on the whole sample (`SSR_pooled`) and on the two segments (`SSR_1`, `SSR_2`). With `k = 2` parameters and `n` rows:

`F = ((SSR_pooled − SSR_1 − SSR_2) / k) / ((SSR_1 + SSR_2) / (n − 2k))`, p-value from `F(k, n − 2k)`.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def chow_test(y, x, break_idx):
    y, x = np.asarray(y, float), np.asarray(x, float)

    def ssr(yy, xx):
        return sm.OLS(yy, sm.add_constant(xx)).fit().ssr

    n, k = y.size, 2
    s1, s2 = ssr(y[:break_idx], x[:break_idx]), ssr(y[break_idx:], x[break_idx:])
    F = ...                                               # ✍️
    return float(F), float(f_dist.sf(F, k, n - 2 * k))

bp, good = p.breaking_pair(), p.cointegrated_pair()        # bp dies at day 900; good never does
rng = np.random.default_rng(1)
xi = rng.normal(size=1500)
yi = 1 + 2 * xi + rng.normal(size=1500)                    # a textbook regression with iid errors
cases = {"breaking pair, break at 900": (bp.y, bp.x, 900), "healthy pair, 'break' at 750": (good.y, good.x, 750),
         "iid regression, 'break' at 750": (yi, xi, 750)}
mine = {k: p.attempt(chow_test, *c) for k, c in cases.items()}
mine = p.check("chow_test", mine, {k: p.chow_test(*c) for k, c in cases.items()})
pd.DataFrame(mine, index=["F", "p-value"]).T

The real break is unmistakable, and the iid regression correctly shows none. But the **healthy** pair also "breaks" at the 5% level. The Chow test assumes independent errors; a cointegrated spread is autocorrelated by design (that is the whole trade), so the test is over-confident. CUSUM, for a break at an **unknown** date, has the same problem:

In [ ]:
print(f"CUSUM p-value — breaking pair: {p.cusum_pvalue(bp.y, bp.x):.1e}, healthy pair: {p.cusum_pvalue(good.y, good.x):.1e}, "
      f"iid regression: {p.cusum_pvalue(yi, xi):.2f}")
false_alarms = sum(p.cusum_pvalue(g.y, g.x) < 0.05 for g in (p.cointegrated_pair(seed=s) for s in range(100)))
print(f"100 healthy cointegrated pairs: CUSUM rejects stability for {false_alarms} of them")

## 2. Monitor instead

A practical alternative: re-run Engle–Granger every month (21 days) on the last 250 days and watch the p-value, the half-life, the hedge ratio and the spread volatility (`p.rolling_stability`). The vertical line is the true break.

In [ ]:
stab = p.rolling_stability(bp.y, bp.x)
fig, axes = plt.subplots(1, 4, figsize=(14, 3))
for ax, col in zip(axes, stab.columns):
    ax.plot(stab.index, stab[col]); ax.axvline(bp.index[900], color="black", lw=1); ax.set_title(col)
    ax.tick_params(axis="x", labelrotation=45)
axes[1].set_yscale("log")
plt.tight_layout(); plt.show()

Turn the dashboard into a rule. A check is **broken** when `pvalue > max_p`, or `half_life > max_hl`, or the hedge ratio has drifted more than `max_beta_drift` from the first check (`|β/β_first − 1|`). Retire the pair at the first check where the rule has been broken on `patience` **consecutive** checks; return `None` if it never is.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def retirement_date(stab, max_p=0.2, max_hl=30, max_beta_drift=0.25, patience=2):
    bad = (...).to_numpy()                                # ✍️ True where any of the three rules is broken
    run = 0
    for date, b in zip(stab.index, bad):
        run = run + 1 if b else 0
        if run >= patience:
            return date
    return None

stab_good = p.rolling_stability(good.y, good.x)
cases = [(stab,), (stab_good,), (stab, 0.2, 30, 0.25, 1), (stab_good, 0.05)]
mine = [p.attempt(retirement_date, *c) for c in cases]
mine = p.check("retirement_date", mine, [p.retirement_date(*c) for c in cases])
pd.Series(mine, index=["breaking pair", "healthy pair", "breaking pair, patience 1", "healthy pair, max p 0.05"],
          name="retired on").to_frame()

The rule catches the dead pair within one 250-day window of the break, and leaves the healthy pair alone. Tighten it and it fires on one bad month of a healthy relationship: with patience 1, the breaking pair is retired in March 2017, more than two years *before* its break; with `max_p` 0.05, the healthy pair is retired in February 2017.

## 3. What retirement saves

Trade the breaking pair as in notebook 03 (formation 750 days, rolling z, which never stops out), then again with the position forced to flat from the retirement date on.

In [ ]:
F = 750
eb = p.engle_granger(bp.y.iloc[:F], bp.x.iloc[:F])
z = p.rolling_zscore(bp.y - eb["beta"] * bp.x - eb["alpha"], round(3 * eb["half_life"]))
pos = p.pairs_positions(z)
cut = bp.index.get_loc(mine[0])
pos_retired = np.where(np.arange(len(pos)) >= cut, 0.0, pos)
pl, pl_retired = p.pair_pnl(pos, bp.y, bp.x, eb["beta"]), p.pair_pnl(pos_retired, bp.y, bp.x, eb["beta"])
print(f"break at day 900, retired at day {cut}; P&L after the break: {pl[900:].sum():+.1%} without retirement, "
      f"{pl_retired[900:].sum():+.1%} with it")

## 4. The best of many strategies

Try 50 strategies and report the best one's t-test, and you will almost always find "significance". **White's Reality Check** asks the right question: is the *best* mean larger than what the best of 50 would show by luck?

* `V = max_k √T·mean_k` (the best strategy's scaled mean).
* Each bootstrap draws one set of rows for **all** strategies (`p.stationary_bootstrap_indices(T, mean_block, rng)`, which keeps short-range dependence), and records `V* = max_k √T·(mean*_k − mean_k)`: the best of 50 when every true mean is zero.
* p-value = share of `V* >= V`.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
from scipy.stats import norm

def reality_check(perf, n_boot=500, mean_block=10, seed=0):
    P = np.asarray(perf, dtype=float)
    T = P.shape[0]
    m = P.mean(axis=0)
    V = ...                                               # ✍️
    rng = np.random.default_rng(seed)
    Vs = np.array([... for _ in range(n_boot)])           # ✍️ one V* per bootstrap
    best = int(np.argmax(m))
    t = m[best] / (P[:, best].std(ddof=1) / np.sqrt(T))
    return {"best": best, "p_value": float((Vs >= V).mean()), "naive_p": float(norm.sf(t))}

noise = np.random.default_rng(0).normal(0, 0.01, (1000, 50))     # 50 strategies, 4 years, no edge at all
edge = noise.copy()
edge[:, 7] += 0.0012                                              # strategy 7 has a real edge (Sharpe ≈ 1.9)
mine = [p.attempt(reality_check, noise), p.attempt(reality_check, edge)]
mine = p.check("reality_check", mine, [p.reality_check(noise), p.reality_check(edge)])
pd.DataFrame(mine, index=["50 noise strategies", "one has a real edge"]).round(4)

The best of 50 noise strategies has a naive p-value below 1%. The Reality Check says 0.24: exactly what you'd expect from the best of 50 coins. With a real edge in strategy 7, both agree. Keep a log of *every* variant you try (Part 8's research log) so the "50" is honest.

## Wrap-up

* Chow and CUSUM assume iid errors; on a cointegrated spread they cry wolf.
* Monitor rolling p-value, half-life, β and spread volatility; retire pairs by rule, not by feel.
* Correct for every strategy you tried: FDR for many pairs, the Reality Check (or Hansen's SPA) for the best of many.
* Graded version: `labs/part09/week32_statarb` (`chow_test`, `cusum_pvalue`, `rolling_stability`, `retirement_date`, `reality_check`).